# Notebook 1 — Data Preparation (nuScenes v1.0-trainval)

## Kaggle setup
1. Add datasets: **mariaamm/nuscenes-trainval-metadata** and **adecours/nuscenes-fulldataset-1-510-trainval**
2. Settings → Accelerator → GPU T4 ×2
3. Run all cells. Cell 2 takes ~15 min, Cell 7 takes ~2 hrs.
4. After completion: Save Version → Output tab → "New Dataset from Output" → name it e.g. `intentformer-nb1-out`

## Colab setup
1. Mount Google Drive
2. Run Cell 1 first — it installs kagglehub and downloads both datasets
3. Outputs are saved to your Drive at `MyDrive/intentformer/`

In [ ]:
# ── Cell 0: Detect environment + install deps ─────────────────────────────────
import os, sys, subprocess

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", *args, "-q"], check=True)

IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Other'}")

pip("nuscenes-devkit", "--no-deps")
pip("pyquaternion", "cachetools", "fire")

if IS_COLAB:
    pip("kagglehub")
    print("✓ kagglehub installed for Colab")

print("✓ Dependencies ready")


In [ ]:
# ── Cell 1: Set paths for Kaggle or Colab ────────────────────────────────────
import os, sys

IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB  = not IS_KAGGLE

if IS_KAGGLE:
    # Kaggle: datasets are mounted at /kaggle/input/ automatically
    # Outputs go to /kaggle/working/ — save this as a dataset after NB1 finishes
    EXTRACT_TO  = "/kaggle/working/nuscenes"
    OUTPUT_DIR  = "/kaggle/working"
    DATA_SEARCH = "/kaggle/input"
    print("Running on Kaggle")
    print("Attach: mariaamm/nuscenes-trainval-metadata")
    print("Attach: adecours/nuscenes-fulldataset-1-510-trainval")

else:
    # Colab: mount Drive first, then download datasets via kagglehub
    from google.colab import drive
    drive.mount("/content/drive")

    EXTRACT_TO  = "/content/drive/MyDrive/intentformer/nuscenes"
    OUTPUT_DIR  = "/content/drive/MyDrive/intentformer"
    DATA_SEARCH = None   # set after kagglehub downloads below
    print("Running on Colab")

os.makedirs(f"{EXTRACT_TO}/samples/LIDAR_TOP", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"EXTRACT_TO : {EXTRACT_TO}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")


In [ ]:
# ── Cell 2: Download datasets (Colab only) / locate files (Kaggle) ───────────
# Kaggle: files are already at /kaggle/input/ — nothing to download
# Colab:  kagglehub downloads to ~/.cache/kagglehub/datasets/...

import glob, os

if IS_KAGGLE:
    # On Kaggle just search the input directory
    search_roots = ["/kaggle/input"]

else:
    # Colab: download both datasets via kagglehub
    # You must have kaggle.json in ~/.kaggle/ OR set KAGGLE_USERNAME / KAGGLE_KEY env vars
    # Get your API key from kaggle.com → Settings → API → Create New Token
    import kagglehub

    print("Downloading metadata dataset (~1 GB)...")
    meta_path = kagglehub.dataset_download("mariaamm/nuscenes-trainval-metadata")
    print(f"  → {meta_path}")

    print("Downloading LiDAR blobs dataset (~150 GB, takes a while)...")
    blob_path = kagglehub.dataset_download("adecours/nuscenes-fulldataset-1-510-trainval")
    print(f"  → {blob_path}")

    search_roots = [meta_path, blob_path]

# ── Find the actual files ─────────────────────────────────────────────────────
def find_file(filename, roots):
    for root in roots:
        matches = glob.glob(f"{root}/**/{filename}", recursive=True)
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Cannot find {filename} under {roots}\n"
        f"Check that both datasets are attached (Kaggle) or downloaded (Colab)."
    )

META_TGZ = find_file("v1.0-trainval_meta.tgz", search_roots)
print(f"\n✓ Metadata tgz : {META_TGZ}")

BLOB_FILES = []
for i in range(1, 6):
    try:
        p = find_file(f"v1.0-trainval0{i}_blobs.tgz", search_roots)
        BLOB_FILES.append(p)
        print(f"✓ Blob {i}       : {p}")
    except FileNotFoundError as e:
        print(f"  WARNING: blob {i} not found — {e}")

assert len(BLOB_FILES) > 0, "No blob files found. Check dataset is attached/downloaded."
print(f"\nFound {len(BLOB_FILES)}/5 blob files.")


In [ ]:
# ── Cell 3: Extract metadata + LiDAR ─────────────────────────────────────────
import tarfile

# ── Metadata ──────────────────────────────────────────────────────────────────
if not os.path.exists(f"{EXTRACT_TO}/v1.0-trainval/scene.json"):
    print("Extracting metadata (~1 min)...", flush=True)
    with tarfile.open(META_TGZ, "r:gz") as tar:
        tar.extractall(EXTRACT_TO, filter="data")
    print("✓ Metadata extracted.")
else:
    print("✓ Metadata already present, skipping.")

# ── LiDAR .pcd.bin files ──────────────────────────────────────────────────────
print("\nExtracting LiDAR .pcd.bin files (first run ~10-15 min)...", flush=True)
extracted = skipped = 0
for blob_path in BLOB_FILES:
    print(f"  {os.path.basename(blob_path)}...", end=" ", flush=True)
    with tarfile.open(blob_path, "r:gz") as tar:
        for member in tar:
            if "samples/LIDAR_TOP" not in member.name: continue
            if not member.name.endswith(".pcd.bin"):    continue
            dst = f"{EXTRACT_TO}/samples/LIDAR_TOP/{os.path.basename(member.name)}"
            if os.path.exists(dst): skipped += 1; continue
            with tar.extractfile(member) as src_f, open(dst, "wb") as dst_f:
                dst_f.write(src_f.read())
            extracted += 1
    print("done")

print(f"\n✓ Extracted {extracted:,} new files, skipped {skipped:,} existing.")
lidar_count = len(glob.glob(f"{EXTRACT_TO}/samples/LIDAR_TOP/*.pcd.bin"))
print(f"✓ Total LiDAR files on disk: {lidar_count:,}")


In [ ]:
# ── Cell 4: Imports + config ──────────────────────────────────────────────────
import numpy as np, pandas as pd, pickle, json, random
from collections import defaultdict
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.splits import create_splits_scenes

NUSCENES_ROOT    = EXTRACT_TO
NUSCENES_VERSION = "v1.0-trainval"

PAST_FRAMES   = 4
FUTURE_FRAMES = 6
WINDOW_LEN    = PAST_FRAMES + FUTURE_FRAMES
DT            = 0.5

TARGET_CATEGORIES = ("human.pedestrian", "vehicle.bicycle")
NEIGHBOR_RADIUS   = 20.0
N_MAX_NEIGHBORS   = 5
SEED              = 42
random.seed(SEED); np.random.seed(SEED)

print(f"Loading {NUSCENES_VERSION} from {NUSCENES_ROOT} ...")
nusc = NuScenes(version=NUSCENES_VERSION, dataroot=NUSCENES_ROOT, verbose=False)
print(f"✓ Loaded {len(nusc.scene)} scenes | {len(nusc.sample):,} samples")


In [ ]:
# ── Cell 5: Scene split ───────────────────────────────────────────────────────
raw_splits     = create_splits_scenes()
train_scenes   = set(raw_splits["train"])
val_scene_list = raw_splits["val"]
val_scenes     = set(val_scene_list[:100])
test_scenes    = set(val_scene_list[100:])

scene_name_map = {s["token"]: s["name"] for s in nusc.scene}

def split_for_scene(scene_token):
    name = scene_name_map[scene_token]
    if name in train_scenes: return "train"
    if name in val_scenes:   return "val"
    return "test"

assert not (train_scenes & val_scenes),  "train/val overlap"
assert not (train_scenes & test_scenes), "train/test overlap"
assert not (val_scenes   & test_scenes), "val/test overlap"
assert len(test_scenes) > 0
print(f"train: {len(train_scenes)} | val: {len(val_scenes)} | test: {len(test_scenes)}")
print("Overlap check: PASS")


In [ ]:
# ── Cell 6: Helper functions ──────────────────────────────────────────────────
def get_category(nusc, instance_token):
    inst = nusc.get("instance", instance_token)
    return nusc.get("category", inst["category_token"])["name"]

def is_target(cat):
    return any(cat.startswith(c) for c in TARGET_CATEGORIES)

def to_agent_centric(xy_global, origin, heading_vec):
    translated = xy_global - origin
    angle = np.arctan2(heading_vec[0], heading_vec[1])
    c, s  = np.cos(angle), np.sin(angle)
    R     = np.array([[c, -s], [s, c]])
    return (R @ translated.T).T

def infer_intent_from_past(past_xy_ac):
    speed   = np.linalg.norm(np.diff(past_xy_ac, axis=0), axis=1).mean() / DT
    if speed < 0.3: return "waiting"
    diffs   = np.diff(past_xy_ac, axis=0)
    hdgs    = np.arctan2(diffs[:, 1], diffs[:, 0])
    ang_vel = np.abs(np.diff(hdgs)).mean() if len(hdgs) > 1 else 0.0
    disp    = past_xy_ac[-1] - past_xy_ac[0]
    if ang_vel > 0.3:                       return "turning"
    if abs(disp[0]) > abs(disp[1]) * 1.5:  return "crossing"
    if speed < 0.5:                         return "stopping"
    return "walking_straight"

INTENT2IDX = {"waiting":0, "crossing":1, "turning":2, "stopping":3, "walking_straight":4}

def extract_lidar_context(nusc, sample_token, agent_xy_global, radius=10.0):
    try:
        sd_tok  = nusc.get("sample", sample_token)["data"]["LIDAR_TOP"]
        sd      = nusc.get("sample_data", sd_tok)
        pc_path = os.path.join(nusc.dataroot, sd["filename"])
        pc      = np.fromfile(pc_path, dtype=np.float32).reshape(-1, 5)
        pts     = pc[:, :2]
        dists   = np.linalg.norm(pts - agent_xy_global, axis=1)
        mask    = dists < radius
        nearby  = pc[mask]
        if len(nearby) < 3:
            return np.zeros(6, dtype=np.float32), False
        feat = np.array([
            len(nearby) / 500.0,
            nearby[:, 2].mean(),
            nearby[:, 2].std() + 1e-6,
            nearby[:, 3].mean(),
            dists[mask].mean() / radius,
            dists[mask].std() / radius + 1e-6,
        ], dtype=np.float32)
        return feat, True
    except Exception:
        return np.zeros(6, dtype=np.float32), False

print("✓ Helpers defined.")


In [ ]:
# ── Cell 7: Pass 1 — global tracks ───────────────────────────────────────────
inst_tracks = defaultdict(lambda: defaultdict(list))

for scene in nusc.scene:
    stok = scene["token"]
    tok  = scene["first_sample_token"]
    fidx = 0
    while tok:
        s = nusc.get("sample", tok)
        for ann_token in s["anns"]:
            ann      = nusc.get("sample_annotation", ann_token)
            inst_tok = ann["instance_token"]
            cat      = get_category(nusc, inst_tok)
            if not is_target(cat): continue
            x, y = ann["translation"][0], ann["translation"][1]
            inst_tracks[stok][inst_tok].append((fidx, x, y, cat, s["token"]))
        tok  = s["next"]
        fidx += 1

total = sum(len(v) for sc in inst_tracks.values() for v in sc.values())
print(f"Scenes with targets : {len(inst_tracks)}")
print(f"Agent-frame entries : {total:,}")


In [ ]:
# ── Cell 8: Pass 2 — sliding windows ─────────────────────────────────────────
from tqdm.auto import tqdm

traj_records  = []
neighbors_map = {}
lidar_map     = {}

for scene in tqdm(nusc.scene, desc="Scenes"):
    stok      = scene["token"]
    split_tag = split_for_scene(stok)

    frame_agents = defaultdict(list)
    for inst_tok, frames in inst_tracks[stok].items():
        for fidx, x, y, cat, stk in frames:
            frame_agents[fidx].append((inst_tok, x, y))

    for inst_tok, frames in inst_tracks[stok].items():
        frames.sort(key=lambda f: f[0])

        for start in range(len(frames) - WINDOW_LEN + 1):
            window  = frames[start: start + WINDOW_LEN]
            indices = [w[0] for w in window]
            if indices[-1] - indices[0] != WINDOW_LEN - 1:
                continue

            sid       = f"{inst_tok}_{start}"
            past_xy_g = np.array([[w[1], w[2]] for w in window[:PAST_FRAMES]])
            fut_xy_g  = np.array([[w[1], w[2]] for w in window[PAST_FRAMES:]])

            origin      = past_xy_g[-1]
            raw_heading = past_xy_g[-1] - past_xy_g[-2]
            heading_vec = (raw_heading if np.linalg.norm(raw_heading) > 1e-3
                           else np.array([0.0, 1.0]))
            past_ac = to_agent_centric(past_xy_g, origin, heading_vec)
            fut_ac  = to_agent_centric(fut_xy_g,  origin, heading_vec)
            intent_idx = INTENT2IDX[infer_intent_from_past(past_ac)]

            neigh_arr  = np.zeros((N_MAX_NEIGHBORS, PAST_FRAMES, 4), np.float32)
            slot       = 0
            seen_slots = {}
            for rel_t, abs_fi in enumerate(indices[:PAST_FRAMES]):
                for other_tok, ox, oy in frame_agents[abs_fi]:
                    if other_tok == inst_tok: continue
                    dist = np.linalg.norm([ox - origin[0], oy - origin[1]])
                    if dist > NEIGHBOR_RADIUS: continue
                    if other_tok not in seen_slots:
                        if slot >= N_MAX_NEIGHBORS: continue
                        seen_slots[other_tok] = slot; slot += 1
                    s = seen_slots[other_tok]
                    o_ac = to_agent_centric(np.array([[ox, oy]]), origin, heading_vec)[0]
                    neigh_arr[s, rel_t, :2] = o_ac
                    if rel_t > 0:
                        neigh_arr[s, rel_t, 2:] = (o_ac - neigh_arr[s, rel_t-1, :2]) / DT
            neighbors_map[sid] = neigh_arr

            sample_tok_mid = window[PAST_FRAMES - 1][4]
            feat, valid    = extract_lidar_context(nusc, sample_tok_mid, origin)
            lidar_map[sid] = {"feat": feat, "valid": valid}

            for rel_t in range(WINDOW_LEN):
                ac_xy = past_ac[rel_t] if rel_t < PAST_FRAMES else fut_ac[rel_t - PAST_FRAMES]
                traj_records.append({
                    "sample_id": sid, "scene_token": stok,
                    "split": split_tag, "category": window[0][3],
                    "frame_rel": rel_t,
                    "x": float(ac_xy[0]), "y": float(ac_xy[1]),
                    "intent_idx": intent_idx,
                })

df = pd.DataFrame(traj_records)
print(f"Windows: {df['sample_id'].nunique():,}")
print(df[df['frame_rel']==0].groupby('split').size().to_string())


In [ ]:
# ── Cell 9: Normalisation ─────────────────────────────────────────────────────
train_sids = set(df[df["split"] == "train"]["sample_id"].unique())
all_diffs  = []
for sid, sdf in df.groupby("sample_id"):
    if sid not in train_sids: continue
    vals = sdf.sort_values("frame_rel")[["x","y"]].values
    all_diffs.append(np.diff(vals, axis=0))

all_diffs = np.concatenate(all_diffs, axis=0)
std_x = max(float(np.std(all_diffs[:, 0])), 0.01)
std_y = max(float(np.std(all_diffs[:, 1])), 0.01)
print(f"std_x: {std_x:.4f} m  |  std_y: {std_y:.4f} m")

df["x_norm"] = df["x"] / std_x
df["y_norm"] = df["y"] / std_y

for sid in neighbors_map:
    neighbors_map[sid][:, :, 0] /= std_x
    neighbors_map[sid][:, :, 1] /= std_y
    neighbors_map[sid][:, :, 2] /= std_x
    neighbors_map[sid][:, :, 3] /= std_y


In [ ]:
# ── Cell 10: Save + sanity checks ────────────────────────────────────────────
pd.Series({"std_x": std_x, "std_y": std_y}).to_csv(f"{OUTPUT_DIR}/norm_stats.csv")
df.to_csv(f"{OUTPUT_DIR}/trajectories.csv", index=False)
with open(f"{OUTPUT_DIR}/neighbors.pkl",   "wb") as fh: pickle.dump(neighbors_map, fh)
with open(f"{OUTPUT_DIR}/lidar_feats.pkl", "wb") as fh: pickle.dump(lidar_map,     fh)

print(f"Outputs written to: {OUTPUT_DIR}")
print(f"  norm_stats.csv    std_x={std_x:.4f}  std_y={std_y:.4f}")
print(f"  trajectories.csv  {len(df):,} rows")
print(f"  neighbors.pkl     {len(neighbors_map):,} entries")
print(f"  lidar_feats.pkl   {len(lidar_map):,} entries")

leaked = (df[df["frame_rel"]==0]
          .groupby("scene_token")["split"].nunique()
          .pipe(lambda s: s[s > 1]))
print(f"Scene-leakage check : {'PASS' if len(leaked)==0 else 'FAIL'}")

n_test = int((df[df["frame_rel"]==0]["split"] == "test").sum())
assert n_test > 0, "Test split is empty."
print(f"Test split          : {n_test:,} windows ✓")

n_valid = sum(1 for v in lidar_map.values() if v["valid"])
print(f"LiDAR valid         : {n_valid:,}/{len(lidar_map):,} ({n_valid/len(lidar_map)*100:.1f}%)")

if IS_KAGGLE:
    print("\n→ Kaggle: Save Version → Output tab → New Dataset from Output")
    print("  Name it e.g. 'intentformer-nb1-out', then attach to NB2.")
else:
    print(f"\n→ Colab: outputs saved to Drive at {OUTPUT_DIR}")
    print("  NB2 INPUT_DIR should point to the same path.")
